# Chapter 10: Machine Learning Integration for Agentic Systems

Hands-On: The AI Health Copilot, End to End Through Every CRISP-DM Phase

Extracted from: chapter_10_ml_integration.md
Source book: Agentic AI: Building AI Agents and Retrieval Systems,
a Masterclass in LLM Agents, RAG, and Production Deployment.

Every block below was verified by direct execution before being
written into the handbook; run this file top to bottom, or copy
out the section you need. Where a step needs an API key
(OPENAI_API_KEY / ANTHROPIC_API_KEY), it is loaded from a local
.env file via python-dotenv, following Chapter 5's own security
discipline, never hardcoded.

NOTE: this file calls a real LLM API and needs a valid API key
exported as an environment variable before it will run end to end.

## Installation

Run this once per environment before the cells below.

In [1]:
%pip install -q pandas numpy scikit-learn openai pydantic python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
load_dotenv()  # picks up OPENAI_API_KEY / ANTHROPIC_API_KEY / COHERE_API_KEY from the project's .env


True

**Step 1: Load the data and look at its basic shape before doing anything else.**

In [3]:
import pandas as pd

df = pd.read_csv("dataset/diabetes.csv")
print(df.shape)
print(df.head())
print(df["Outcome"].value_counts())

(768, 9)
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
Outcome
0    500
1    268
Name: count, dtype: int64


**Step 2: Audit for a specific, well-documented data quality problem before trusting any statistic.**

In [4]:
cols_to_check = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in cols_to_check:
    zero_count = (df[col] == 0).sum()
    print(f"{col}: {zero_count} zero values ({zero_count / len(df):.1%})")

Glucose: 5 zero values (0.7%)
BloodPressure: 35 zero values (4.6%)
SkinThickness: 227 zero values (29.6%)
Insulin: 374 zero values (48.7%)
BMI: 11 zero values (1.4%)


**Step 3: Convert the invalid zeros into genuine, explicit missing values.**

In [5]:
import numpy as np

df[cols_to_check] = df[cols_to_check].replace(0, np.nan)
print(df.isnull().sum())

Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64


**Step 4: Split into training and test sets before any further preparation, not after.**

In [6]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Outcome"])
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

**Step 5: Impute and scale inside one `Pipeline`, following this chapter's own best-practice rule from earlier.**

In [7]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

preprocessing = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

**Step 6: Train two candidate models inside the same pipeline shape, so they are genuinely comparable.**

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)),
])
rf_pipeline.fit(X_train, y_train)

logreg_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
logreg_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Pregnancies','Glucose','BloodPressure',...,'BMI', 'DiabetesPedigreeFunction','Age']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For str

**Step 7: Score both models on the held-out test set, never seen during training.**

In [9]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

for name, pipeline in [("Random Forest", rf_pipeline), ("Logistic Regression", logreg_pipeline)]:
    pred = pipeline.predict(X_test)
    proba = pipeline.predict_proba(X_test)[:, 1]
    print(f"--- {name} ---")
    print("Accuracy:", round(accuracy_score(y_test, pred), 4))
    print("AUC:", round(roc_auc_score(y_test, proba), 4))
    print(confusion_matrix(y_test, pred))

--- Random Forest ---
Accuracy: 0.7403
AUC: 0.8078
[[85 15]
 [25 29]]
--- Logistic Regression ---
Accuracy: 0.7078
AUC: 0.813
[[82 18]
 [27 27]]


**Step 8: Check the result is not a lucky split with five-fold cross-validation.**

In [10]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf_pipeline, X, y, cv=5, scoring="roc_auc")
print("AUC across 5 folds:", [round(s, 3) for s in scores])
print("Mean AUC:", round(scores.mean(), 4), "Std:", round(scores.std(), 4))

AUC across 5 folds: [0.819, 0.795, 0.849, 0.893, 0.84]
Mean AUC: 0.8393 Std: 0.0327


**Step 8b: Before accepting `max_depth=5, n_estimators=200` as final, search a small grid of alternatives properly, using the same 5-fold discipline Step 8 just validated, rather than picking hyperparameters by hand and hoping.**

In [11]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [3, 5, 7],
}

grid_search = GridSearchCV(rf_pipeline, param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
grid_search.fit(X_train, y_train)

print("best params:", grid_search.best_params_)
print("best CV AUC:", round(grid_search.best_score_, 4))

best params: {'model__max_depth': 7, 'model__n_estimators': 100}
best CV AUC: 0.8384


**Step 9: Save the fitted pipeline exactly as the real project's own `.sav` files were produced.**

In [12]:
import pickle

with open("saved_models/diabetes_model.sav", "wb") as f:
    pickle.dump(rf_pipeline, f)

**Step 10: Expose a genuine probability, not a bare label, so downstream logic has something to reason about.**

In [13]:
def predict_diabetes_proba(patient_data: list) -> float:
    return rf_pipeline.predict_proba([patient_data])[0][1]

**Step 11: Deterministically flag which specific features are outside a normal clinical range, in code, before the LLM ever sees the case.**

In [14]:
NORMAL_RANGES = {
    "Glucose": (70, 99), "BloodPressure": (60, 80),
    "BMI": (18.5, 24.9), "Age": (0, 120),
}

def flag_abnormal_features(patient_data: list, feature_names: list) -> list:
    flags = []
    for name, value in zip(feature_names, patient_data):
        if name in NORMAL_RANGES:
            low, high = NORMAL_RANGES[name]
            if value < low or value > high:
                flags.append(f"{name} = {value} (normal range: {low}-{high})")
    return flags

**Step 12: Feed only the deterministic flags, not the raw feature vector, into a structured, role-based, chain-of-thought prompt.**

In [15]:
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()

class ClinicalExplanation(BaseModel):
    risk_level: str
    key_factors: list[str]
    recommendation: str
    disclaimer: str

def explain_diagnosis(probability: float, flags: list) -> ClinicalExplanation:
    system_prompt = (
        "You are a clinical communication assistant, not a diagnosing physician. "
        "Reason step by step using ONLY the flagged factors provided, never invent "
        "a factor not listed. Think first, then produce your final answer as the "
        "requested structured fields."
    )
    user_prompt = (
        f"Model-predicted diabetes probability: {probability:.0%}.\n"
        f"Flagged out-of-range factors: {flags if flags else 'none flagged'}.\n"
        f"Step 1, reason about what these specific flags suggest. "
        f"Step 2, state a risk_level (low, moderate, high). "
        f"Step 3, list key_factors drawn only from the flags above. "
        f"Step 4, give one plain-language recommendation. "
        f"Step 5, include a disclaimer that this is not a medical diagnosis."
    )
    response = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_prompt}],
        response_format=ClinicalExplanation,
    )
    return response.choices[0].message.parsed

**Step 13: Chain every phase into one callable pipeline, wrapped exactly as Chapter 6's tool-registry pattern requires.**

In [16]:
FEATURE_NAMES = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
                  "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]

def run_health_copilot(patient_data: list) -> dict:
    probability = predict_diabetes_proba(patient_data)
    flags = flag_abnormal_features(patient_data, FEATURE_NAMES)
    explanation = explain_diagnosis(probability, flags)
    return {"probability": probability, "explanation": explanation.model_dump()}

patient = [2, 148, 72, 35, 155, 33.6, 0.627, 50]  # a real, positive-outcome row from the dataset
print(run_health_copilot(patient))

C:\Users\harpa\Documents\Others\PC\project_management\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


{'probability': 0.6654161382623356, 'explanation': {'risk_level': 'high', 'key_factors': ['Glucose = 148 (normal range: 70-99)', 'BMI = 33.6 (normal range: 18.5-24.9)'], 'recommendation': 'Consider consulting a healthcare provider for further diabetes screening and lifestyle advice.', 'disclaimer': 'This assessment is not a medical diagnosis. Please consult a healthcare professional for a definitive diagnosis and personalized advice.'}}
